In [1]:
# 定义10条评估 queries
eval_queries = [
    "How should an organization respond to a phishing email?",
    "What technical controls can prevent phishing attacks?",
    "How can employees identify spear-phishing attempts?",
    "What is DMARC and how does it help prevent email spoofing?",
    "How should incident response teams handle a phishing-related data breach?",
    "What are best practices for email authentication in enterprises?",
    "How can organizations train employees to recognize phishing?",
    "What role does SPF play in email security?",
    "How should a mid-sized enterprise handle a ransomware attack initiated by phishing?",
    "What are the indicators of compromise in a phishing attack?"
]

print(f"Total evaluation queries: {len(eval_queries)}")

Total evaluation queries: 10


In [2]:
# 跑完整 RAG pipeline，收集所有结果
# 复制上一个notebook的函数（这样evaluation notebook独立可运行）
import os, requests
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(
    "../data/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

def query_mistral(prompt: str) -> str:
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "mistral", "prompt": prompt, "stream": False}
    )
    return response.json()["response"]

def rag_pipeline(query: str, k: int = 3) -> dict:
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])
    sources = [f"{doc.metadata.get('source','?')} p.{doc.metadata.get('page','?')}" 
               for doc in docs]
    prompt = f"""You are a cybersecurity advisor helping practitioners.
Use ONLY the context below to answer the question.
If the context does not contain enough information, say so.

Context:
{context}

Question: {query}

Provide exactly 3 concise, actionable guidelines for cybersecurity practitioners.
Format as:
Guideline 1: ...
Guideline 2: ...
Guideline 3: ...
"""
    answer = query_mistral(prompt)
    return {
        "query": query,
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": [doc.page_content for doc in docs]
    }

# 跑所有queries
print("Running RAG pipeline for all queries...")
results = []
for i, query in enumerate(eval_queries):
    print(f"  [{i+1}/10] {query[:50]}...")
    result = rag_pipeline(query)
    results.append(result)

print("\nDone! All 10 queries completed.")

/var/folders/tk/x43kyqbn33v_ttx5rckk29cm0000gn/T/ipykernel_80923/3499837493.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Running RAG pipeline for all queries...
  [1/10] How should an organization respond to a phishing e...
  [2/10] What technical controls can prevent phishing attac...
  [3/10] How can employees identify spear-phishing attempts...
  [4/10] What is DMARC and how does it help prevent email s...
  [5/10] How should incident response teams handle a phishi...
  [6/10] What are best practices for email authentication i...
  [7/10] How can organizations train employees to recognize...
  [8/10] What role does SPF play in email security?...
  [9/10] How should a mid-sized enterprise handle a ransomw...
  [10/10] What are the indicators of compromise in a phishin...

Done! All 10 queries completed.


In [5]:
# 计算 RAGAS 评估指标
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.llms import Ollama
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import Dataset

# 告诉 RAGAS 用 Ollama 而不是 OpenAI
ollama_llm = LangchainLLMWrapper(Ollama(model="mistral"))
hf_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

ragas_data = {
    "question": [r["query"] for r in results],
    "answer": [r["answer"] for r in results],
    "contexts": [r["retrieved_chunks"] for r in results],
}

dataset = Dataset.from_dict(ragas_data)

print("Running RAGAS evaluation with Ollama...")
ragas_results = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=ollama_llm,
    embeddings=hf_embeddings,
)

print("\n=== RAGAS SCORES ===")
print(ragas_results)

/var/folders/tk/x43kyqbn33v_ttx5rckk29cm0000gn/T/ipykernel_80923/1879411188.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/var/folders/tk/x43kyqbn33v_ttx5rckk29cm0000gn/T/ipykernel_80923/1879411188.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
/var/folders/tk/x43kyqbn33v_ttx5rckk29cm0000gn/T/ipykernel_80923/1879411188.py:10: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/var/folders/tk/x43kyqbn33v_ttx5rckk29cm0000gn/T/ipykernel_80923/1879411188.py:11: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  hf_embeddings = LangchainEmbeddingsWrapper(


Running RAGAS evaluation with Ollama...


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Exception raised in Job[0]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[15]: TimeoutError()



=== RAGAS SCORES ===
{'faithfulness': 0.8889, 'answer_relevancy': 0.7961}


In [7]:
import pandas as pd
import json

# 保存 RAGAS 分数表
scores_df = ragas_results.to_pandas()
scores_df.insert(0, "query", [r["query"] for r in results])
scores_df.to_csv("../outputs/ragas_scores.csv", index=False)
print("Scores saved!")
print(scores_df[["query", "faithfulness", "answer_relevancy"]].to_string())

# 保存完整结果
with open("../outputs/full_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nFull results saved!")

Scores saved!
                                                                                 query  faithfulness  answer_relevancy
0                              How should an organization respond to a phishing email?           NaN               NaN
1                                What technical controls can prevent phishing attacks?           NaN               NaN
2                                  How can employees identify spear-phishing attempts?           NaN               NaN
3                           What is DMARC and how does it help prevent email spoofing?           NaN               NaN
4            How should incident response teams handle a phishing-related data breach?           NaN               NaN
5                     What are best practices for email authentication in enterprises?      1.000000               NaN
6                         How can organizations train employees to recognize phishing?           NaN               NaN
7                                 